# Convert the Adelani English→Yoruba model for in-browser use

Converts [`Davlan/m2m100_418M-eng-yor-mt`](https://huggingface.co/Davlan/m2m100_418M-eng-yor-mt) (David Adelani, Masakhane) to ONNX so the Consent-First Kit can run it **in the programme lead's browser** with Transformers.js: free, no server, text never leaves the device.

Run on free Google Colab (CPU is enough, about 10 minutes). Run cells top to bottom.

**Before publishing:** the original model page states no licence. Credit is included below; confirm reuse terms with the author before wide use.

## 1. Install the official Transformers.js conversion tooling

In [ ]:
%cd /content
!rm -rf transformers.js
!git clone --depth 1 --branch 3.8.1 https://github.com/huggingface/transformers.js.git
%cd /content/transformers.js
# Colab's Python has no ready-made wheel for the pinned onnx 1.17; 1.18 is compatible
!sed -i -E 's/^onnx==.*/onnx==1.18.0/' scripts/requirements.txt
!pip install -q -r scripts/requirements.txt
# Colab extras that break the exporter: diffusers needs a newer huggingface_hub, and torch >= 2.9 needs onnxscript.
# torch 2.7 uses the classic, well-tested ONNX exporter (the conversion runs in its own process, so no restart needed).
!pip uninstall -y -q diffusers torchvision torchaudio  # torchvision/torchaudio must match torch; not needed here
!pip install -q torch==2.7.1 onnxscript

## 2. Convert to ONNX, with a quantized copy (about 4x smaller download)

In [ ]:
%cd /content/transformers.js
!python -m scripts.convert --quantize --model_id Davlan/m2m100_418M-eng-yor-mt
!ls -la models/Davlan/m2m100_418M-eng-yor-mt/onnx

## 3. Check the converted model translates before publishing

In [ ]:
from optimum.onnxruntime import ORTModelForSeq2SeqLM
from transformers import AutoTokenizer
path = 'models/Davlan/m2m100_418M-eng-yor-mt'
tok = AutoTokenizer.from_pretrained(path)
mdl = ORTModelForSeq2SeqLM.from_pretrained(path, subfolder='onnx', encoder_file_name='encoder_model.onnx', decoder_file_name='decoder_model_merged.onnx', use_merged=True)
tok.src_lang = 'en'
for line in ['We would like to ask you a few questions about how you have been feeling.',
             'You can say no to any part, and you can change your mind later.']:
    ids = tok(line, return_tensors='pt')
    out = mdl.generate(**ids, forced_bos_token_id=tok.get_lang_id('yo'), max_new_tokens=100)
    print(line, '->', tok.batch_decode(out, skip_special_tokens=True)[0])

## 4. Sign in to Hugging Face
Save a **write** token in Colab Secrets (key icon, left) under the name `Hugging_face_key` and switch on notebook access. The cell reads it by name; never paste the token into a cell.

In [ ]:
# Uses the token saved in Colab Secrets (key icon) under the NAME Hugging_face_key.
# Never type the token itself into a cell.
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('Hugging_face_key'))

## 5. Publish the browser-ready copy with credit to the original author

In [ ]:
from huggingface_hub import HfApi, whoami
REPO = f"{whoami()['name']}/m2m100_418M-eng-yor-mt-onnx"
readme = f'''---
base_model: Davlan/m2m100_418M-eng-yor-mt
library_name: transformers.js
pipeline_tag: translation
language: [en, yo]
---
# m2m100_418M-eng-yor-mt (ONNX, for Transformers.js)

Browser-ready conversion of [Davlan/m2m100_418M-eng-yor-mt](https://huggingface.co/Davlan/m2m100_418M-eng-yor-mt)
by David Adelani (Masakhane), fine-tuned on JW300 and MENYO-20k. All credit to the original authors; no weights were changed.

Used by the [Consent-First Kit](https://github.com/SDBE-Ex/consent-first-kit) to give programme leads a second,
locally made Yoruba **draft** that a native speaker reviews before anything reaches people.

Licence: the original model card states no licence. Reuse terms are being confirmed with the author.
'''
open(f'models/Davlan/m2m100_418M-eng-yor-mt/README.md', 'w').write(readme)
api = HfApi()
api.create_repo(REPO, exist_ok=True)
api.upload_folder(folder_path='models/Davlan/m2m100_418M-eng-yor-mt', repo_id=REPO)
print('Published:', f'https://huggingface.co/{REPO}')
print('Set this in Vercel as VITE_LOCAL_YO_MODEL =', REPO)